# 🤖 AutoML Pilot - Training Template
Use this Colab notebook to run advanced AutoML (PyCaret) on your dataset. This template is designed to work seamlessly with datasets exported from the AutoML Pilot web app.

In [ ]:
# @title 1. Install Dependencies
# Install PyCaret and other necessary libraries
!pip install -q pycaret[full] ydata-profiling pandas
print('✅ Dependencies installed!')

In [ ]:
# @title 2. Email Configuration (Optional)
import smtplib
import ssl
import json
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart

recipient_email = "" # @param {type:"string"}
sender_email = "" # @param {type:"string"}
sender_password = "" # @param {type:"string"}

def send_metrics_email(results_dict, task):
    if not recipient_email or not sender_email or not sender_password:
        print("⚠️ Email credentials not provided. Skipping email report.")
        return
    
    try:
        msg = MIMEMultipart("alternative")
        msg["Subject"] = f"AutoML Pilot Colab Results - {task}"
        msg["From"] = sender_email
        msg["To"] = recipient_email
        
        html = f"""
        <html>
          <body style='font-family: Arial, sans-serif;'>
            <h2 style='color: #7c3aed;'>🚀 AutoML Pilot - Colab Training Report</h2>
            <p>Your model training on Google Colab has completed.</p>
            <table border='1' style='border-collapse: collapse; width: 100%;'>
              <tr style='background-color: #f3f4f6;'>
                <th style='padding: 8px;'>Metric</th><th style='padding: 8px;'>Value</th>
              </tr>
              {''.join([f"<tr><td style='padding: 8px;'>{k}</td><td style='padding: 8px;'>{v}</td></tr>" for k, v in results_dict.items()])}
            </table>
          </body>
        </html>
        """
        msg.attach(MIMEText(html, "html"))
        
        context = ssl.create_default_context()
        with smtplib.SMTP_SSL("smtp.gmail.com", 465, context=context) as server:
            server.login(sender_email, sender_password)
            server.sendmail(sender_email, recipient_email, msg.as_string())
        print("✅ Results emailed successfully!")
    except Exception as e:
        print(f"❌ Failed to send email: {e}")

In [ ]:
# @title 3. Load Dataset
import pandas as pd
import os
from google.colab import files

upload_method = "Direct Upload" # @param ["Direct Upload", "Google Drive"]

if upload_method == "Direct Upload":
    uploaded = files.upload()
    if uploaded:
        filename = list(uploaded.keys())[0]
        df = pd.read_csv(filename)
    else:
        print("❌ No file uploaded.")
else:
    from google.colab import drive
    drive.mount('/content/drive')
    dataset_path = '/content/drive/MyDrive/dataset.csv' # @param {type:"string"}
    if os.path.exists(dataset_path):
        df = pd.read_csv(dataset_path)
    else:
        print(f"❌ File not found at {dataset_path}")

if 'df' in locals():
    print(f'✅ Loaded dataset: {df.shape[0]} rows x {df.shape[1]} columns')
    display(df.head())

In [ ]:
# @title 4. AutoML Setup & Training
from pycaret.classification import setup as cls_setup, compare_models as cls_compare, pull as cls_pull, save_model as cls_save, finalize_model as cls_finalize
from pycaret.regression import setup as reg_setup, compare_models as reg_compare, pull as reg_pull, save_model as reg_save, finalize_model as reg_finalize
from ydata_profiling import ProfileReport

# Configuration
target_column = 'target' # @param {type:"string"}
task_type = 'classification' # @param ["classification", "regression"]
run_eda = True # @param {type:"boolean"}

if run_eda:
    print("📊 Generating EDA Report...")
    profile = ProfileReport(df, minimal=True)
    profile.to_file("eda_report.html")
    print("✅ EDA Report saved as eda_report.html")

print(f'🚀 Starting AutoML for {task_type} on target: {target_column}')

if task_type == 'classification':
    s = cls_setup(data=df, target=target_column, session_id=123, verbose=False, html=False)
    best_model = cls_compare()
    leaderboard = cls_pull()
    metrics = leaderboard.iloc[0].to_dict()
    final_model = cls_finalize(best_model)
    cls_save(final_model, 'best_model')
else:
    s = reg_setup(data=df, target=target_column, session_id=123, verbose=False, html=False)
    best_model = reg_compare()
    leaderboard = reg_pull()
    metrics = leaderboard.iloc[0].to_dict()
    final_model = reg_finalize(best_model)
    reg_save(final_model, 'best_model')

display(leaderboard)
send_metrics_email(metrics, task_type)
print('✅ Model saved as best_model.pkl')

In [ ]:
# @title 5. Download Results
from google.colab import files
import os

for f in ['best_model.pkl', 'eda_report.html']:
    if os.path.exists(f):
        files.download(f)
    elif os.path.exists(f + '.pkl'):
        files.download(f + '.pkl')